In [49]:
from matplotlib import pyplot as plt
import polars as pl

from fantasy_football.constants import TRANSFORMED_DATA_FOLDER, CLUBELO_TO_FPL
from fantasy_football.features.match_form import NO_FORM_COLUMN, PENALTY_EXPOSURE_COLUMN
from fantasy_football.features.views import register_feature_views
from fantasy_football.storage.database import get_connection
from fantasy_football.modelling.folds import TrainTestSplitStrategy
from fantasy_football.modelling.goals import GoalsRatePredictor, EXPERIMENT_NAME, GOALS_SPEC, TRAINING_SEASONS, TRAINING_POSITIONS
from fantasy_football.modelling.points import position_dummy_names

To do:
-  ~~Get the ELO joined to the training data~~
- ~~Have ELO difference as a feature~~
- ~~Train the regular model like this and score it~~
- [ ] Adjust the rolling features a bit and see what you can do (maybe season to date)
- [ ] Add goals in prev season as feature
- [ ] Bring in shot volume features into the model itself and see how this goes
- [ ] Bring touches in oppositon box and see how it does
- [ ] Experiment with different model types for scoring
- [ ] Experiment with hyperparameter optimisation

In [50]:
connection = get_connection()

In [51]:
goals_predictor = GoalsRatePredictor(
    experiment_name=EXPERIMENT_NAME,
    params={},
    model_spec=GOALS_SPEC,
    connection=connection,
    fold_strategy=TrainTestSplitStrategy(
        test_seasons=TRAINING_SEASONS
    ),
)

In [52]:
training_data = goals_predictor.build_training_data()
training_data.head()

1 team_match rows have no kickoff_time and so no position in the rolling window; their fixture is missing from team_fixture.


season,gw,element,opponent,goals_per_90,goals_scored,minutes,is_home,is_defender,is_midfielder,is_forward,has_no_form,goals_scored_per90_rolling_5,xg_per90_rolling_5,xa_per90_rolling_5,expected_pen_attempts_per_90,xg_for_rolling_5,goals_for_rolling_5,xg_against_rolling_5,goals_against_rolling_5
str,i64,i64,i64,f64,i64,i64,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2024-25""",2,277,13,0.0,0,71,false,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.46,0.0,1.02,0.0
"""2024-25""",5,650,14,0.0,0,90,true,1.0,0.0,0.0,0.0,0.0,0.0,0.04,0.016667,1.425,1.0,1.505,1.5
"""2024-25""",6,650,8,0.0,0,90,false,1.0,0.0,0.0,0.0,0.0,0.01,0.02,0.013333,1.348,0.8,1.896,2.8
"""2024-25""",7,650,12,0.0,0,90,true,1.0,0.0,0.0,0.0,0.0,0.06,0.063333,0.011111,1.304,0.8,0.816,0.4
"""2024-25""",8,650,16,0.0,0,90,false,1.0,0.0,0.0,0.0,0.0,0.045,0.095,0.009524,1.164,0.8,1.3,1.0


In [53]:
elo_data_filepath = TRANSFORMED_DATA_FOLDER.joinpath("team_elo.csv")
elo_data = pl.read_csv(elo_data_filepath, try_parse_dates=True)
elo_data.head()


team,elo,from_date,to_date
str,f64,date,date
"""Arsenal""",1841.410278,2016-07-29,2016-08-04
"""Arsenal""",1842.851929,2016-08-05,2016-08-14
"""Arsenal""",1832.216431,2016-08-15,2016-08-16
"""Arsenal""",1833.50293,2016-08-17,2016-08-18
"""Arsenal""",1833.499146,2016-08-19,2016-08-20


In [54]:
CLUBELO_TO_FPL_NAME = {k: v[0] for k, v in CLUBELO_TO_FPL.items()}

elo_data = elo_data.with_columns(
    pl.col("team").replace_strict(
        CLUBELO_TO_FPL_NAME,
        default=None,
        return_dtype=pl.String,
    ).alias("fpl_team")
)
elo_data.head()

team,elo,from_date,to_date,fpl_team
str,f64,date,date,str
"""Arsenal""",1841.410278,2016-07-29,2016-08-04,"""Arsenal"""
"""Arsenal""",1842.851929,2016-08-05,2016-08-14,"""Arsenal"""
"""Arsenal""",1832.216431,2016-08-15,2016-08-16,"""Arsenal"""
"""Arsenal""",1833.50293,2016-08-17,2016-08-18,"""Arsenal"""
"""Arsenal""",1833.499146,2016-08-19,2016-08-20,"""Arsenal"""


In [55]:
from fantasy_football.storage.tables import PLAYER_MATCH, TEAM_FIXTURE, PLAYER_WEEK

player_match = PLAYER_WEEK.load(connection)
player_match = player_match.filter(pl.col("season").is_in(goals_predictor.TRAINING_SEASONS))
player_match.head()

season,gw,element,name,position,team,bonus,minutes,round,total_points,value
str,i64,i64,str,str,str,i64,i64,i64,i64,i64
"""2024-25""",1,1,"""Fábio Ferreira Vieira""","""MID""","""Arsenal""",0,0,1,0,55
"""2024-25""",1,2,"""Gabriel Fernando de Jesus""","""FWD""","""Arsenal""",0,5,1,0,70
"""2024-25""",1,3,"""Gabriel dos Santos Magalhães""","""DEF""","""Arsenal""",0,90,1,6,60
"""2024-25""",1,4,"""Kai Havertz""","""FWD""","""Arsenal""",3,90,1,12,80
"""2024-25""",1,5,"""Karl Hein""","""GK""","""Arsenal""",0,0,1,0,40


In [56]:
team_fixture = TEAM_FIXTURE.load(connection)
team_fixture = team_fixture.filter(pl.col("season").is_in(goals_predictor.TRAINING_SEASONS))
team_fixture.head()

season,gw,team,is_home,opposition,kickoff_time
str,i64,str,bool,str,datetime[μs]
"""2024-25""",1,"""Arsenal""",true,"""Wolves""",2024-08-17 14:00:00
"""2024-25""",1,"""Aston Villa""",false,"""West Ham""",2024-08-17 16:30:00
"""2024-25""",1,"""Bournemouth""",false,"""Nott'm Forest""",2024-08-17 14:00:00
"""2024-25""",1,"""Brentford""",true,"""Crystal Palace""",2024-08-18 13:00:00
"""2024-25""",1,"""Brighton""",false,"""Everton""",2024-08-17 14:00:00


In [57]:
elo_sorted = (
    elo_data
    .with_columns(pl.col("from_date").cast(pl.Datetime("us")))
    .select("team", "elo", "from_date")
    .sort("from_date")
)

elo_fixture = (
    team_fixture
    .sort("kickoff_time")
    .join_asof(
        elo_sorted,
        left_on="kickoff_time",
        right_on="from_date",
        by="team",
        strategy="backward",
    )
)

/var/folders/ws/c0kbfc596sgcz0f3y4dzqtyc0000gn/T/ipykernel_30363/2744409402.py:11: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  .join_asof(


In [58]:
fpl_ids = connection.sql("SELECT * FROM fpl_team_id").pl()

elo_sides = (
    elo_fixture
    .join(fpl_ids, on=["season", "team"], how="left")
    .rename({"team_id": "team_id"})
    .join(
        fpl_ids.rename({"team": "opposition", "team_id": "opposition_id"}),
        on=["season", "opposition"], how="left",
    )
    .select("season", "gw", "is_home", "team_id", "opposition_id", "elo")
)

In [59]:
elo_pairs = (
    elo_sides.rename({"elo": "team_elo"})
    .join(
        elo_sides.select(
            "season", "gw",
            pl.col("team_id").alias("opposition_id"),
            pl.col("elo").alias("opponent_elo"),
        ),
        on=["season", "gw", "opposition_id"], how="left",
    )
    .with_columns((pl.col("team_elo") - pl.col("opponent_elo")).alias("elo_diff"))
)

In [60]:
frame = training_data.join(
    elo_pairs.select("season", "gw", "opposition_id", "is_home",
                     "team_elo", "opponent_elo", "elo_diff"),
    left_on=["season", "gw", "opponent", "is_home"],
    right_on=["season", "gw", "opposition_id", "is_home"],
    how="left",
)
frame.head()

season,gw,element,opponent,goals_per_90,goals_scored,minutes,is_home,is_defender,is_midfielder,is_forward,has_no_form,goals_scored_per90_rolling_5,xg_per90_rolling_5,xa_per90_rolling_5,expected_pen_attempts_per_90,xg_for_rolling_5,goals_for_rolling_5,xg_against_rolling_5,goals_against_rolling_5,team_elo,opponent_elo,elo_diff
str,i64,i64,i64,f64,i64,i64,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2024-25""",2,277,13,0.0,0,71,false,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.46,0.0,1.02,0.0,1565.432983,2055.970947,-490.537964
"""2024-25""",5,650,14,0.0,0,90,true,1.0,0.0,0.0,0.0,0.0,0.0,0.04,0.016667,1.425,1.0,1.505,1.5,1741.469116,1778.970825,-37.501709
"""2024-25""",6,650,8,0.0,0,90,false,1.0,0.0,0.0,0.0,0.0,0.01,0.02,0.013333,1.348,0.8,1.896,2.8,1740.203735,1672.902344,67.301392
"""2024-25""",7,650,12,0.0,0,90,true,1.0,0.0,0.0,0.0,0.0,0.06,0.063333,0.011111,1.304,0.8,0.816,0.4,1740.906128,1938.562866,-197.656738
"""2024-25""",8,650,16,0.0,0,90,false,1.0,0.0,0.0,0.0,0.0,0.045,0.095,0.009524,1.164,0.8,1.3,1.0,1736.295166,1683.698486,52.59668


In [61]:
class Candidate(GoalsRatePredictor):
     FEATURES = GoalsRatePredictor.FEATURES + [
        "elo_diff",
     ]

In [62]:
folds = goals_predictor.fold_strategy.split(frame)
results, summary = Candidate(
    experiment_name=EXPERIMENT_NAME,
    params={},
    model_spec=GOALS_SPEC,
    connection=connection,
    fold_strategy=TrainTestSplitStrategy(
        test_seasons=TRAINING_SEASONS
    ),
).cross_validate(folds)


In [63]:
results

[FoldResult(metrics=GoalsMetrics(poisson_deviance=0.5280227445332648, brier=0.08544508319027913, logloss=0.33352673327601684, base_rate_brier=0.08816830341318986, skill_score=0.030886612506863087, scoring_rate=0.09771689497716896, rate_mae=0.2166128098860984, top_decile_ratio=1.5894338040962839), predictions=shape: (3_285, 8)
 ┌─────────┬─────┬─────────┬──────────┬──────────┬─────────────────┬───────────────┬────────────────┐
 │ season  ┆ gw  ┆ element ┆ opponent ┆ position ┆ predicted_point ┆ actual_points ┆ features       │
 │ ---     ┆ --- ┆ ---     ┆ ---      ┆ ---      ┆ s               ┆ ---           ┆ ---            │
 │ str     ┆ i64 ┆ i64     ┆ i64      ┆ str      ┆ ---             ┆ f64           ┆ str            │
 │         ┆     ┆         ┆          ┆          ┆ f64             ┆               ┆                │
 ╞═════════╪═════╪═════════╪══════════╪══════════╪═════════════════╪═══════════════╪════════════════╡
 │ 2025-26 ┆ 25  ┆ 257     ┆ 6        ┆ DEF      ┆ 0.04     

In [64]:
summary

{'holdout_poisson_deviance': 0.5280227445332648,
 'holdout_brier': 0.08544508319027913,
 'holdout_logloss': 0.33352673327601684,
 'holdout_base_rate_brier': 0.08816830341318986,
 'holdout_skill_score': 0.030886612506863087,
 'holdout_scoring_rate': 0.09771689497716896,
 'holdout_rate_mae': 0.2166128098860984,
 'holdout_top_decile_ratio': 1.5894338040962839}

In [65]:
register_feature_views(connection, rolling_window=10)

1 team_match rows have no kickoff_time and so no position in the rolling window; their fixture is missing from team_fixture.


In [66]:
rolling_features = [
    "goals_scored_per90_",
    "xg_per90_",
    "xa_per90_",
    "xg_for_",
    "goals_for_",
    "xg_against_",
    "goals_against_",
]

rolling_5_features = [
    f"{feature}rolling_5" for feature in rolling_features
]
rolling_10_features = [
    f"{feature}rolling_10" for feature in rolling_features
]

In [67]:
rolling_10_features 

['goals_scored_per90_rolling_10',
 'xg_per90_rolling_10',
 'xa_per90_rolling_10',
 'xg_for_rolling_10',
 'goals_for_rolling_10',
 'xg_against_rolling_10',
 'goals_against_rolling_10']

In [68]:
class Candidate(GoalsRatePredictor):
   PLAYER_FORM_COLUMNS = [
      NO_FORM_COLUMN, 
      "goals_scored_per90_rolling_10",
      "xg_per90_rolling_10", 
      "xa_per90_rolling_10",
      PENALTY_EXPOSURE_COLUMN
     ]
   OWN_TEAM_COLUMNS = ["xg_for_rolling_10", "goals_for_rolling_10"]
   OPPOSITION_COLUMNS = ["xg_against_rolling_10", "goals_against_rolling_10"]
   FEATURES = ["is_home", *position_dummy_names(TRAINING_POSITIONS),
                *PLAYER_FORM_COLUMNS, *OWN_TEAM_COLUMNS, *OPPOSITION_COLUMNS, "elo_diff"]

In [69]:

candidate = Candidate(
    experiment_name=EXPERIMENT_NAME,
    params={},
    model_spec=GOALS_SPEC,
    connection=connection,
    fold_strategy=TrainTestSplitStrategy(
        test_seasons=TRAINING_SEASONS
    ),
)
training_data = candidate.build_training_data()          # now has _rolling_10
frame = training_data.join(                              # re-run cell 12
    elo_pairs.select("season","gw","opposition_id","is_home",
                     "team_elo","opponent_elo","elo_diff"),
    left_on=["season","gw","opponent","is_home"],
    right_on=["season","gw","opposition_id","is_home"],
    how="left",
)
folds = candidate.fold_strategy.split(frame)
results, summary = candidate.cross_validate(folds)

1 team_match rows have no kickoff_time and so no position in the rolling window; their fixture is missing from team_fixture.


BinderException: Binder Error: Values list "mf" does not have a column named "goals_scored_per90_rolling_10"

In [ ]:
folds = goals_predictor.fold_strategy.split(frame)
results, summary = Candidate(
    experiment_name=EXPERIMENT_NAME,
    params={},
    model_spec=GOALS_SPEC,
    connection=connection,
    fold_strategy=TrainTestSplitStrategy(
        test_seasons=TRAINING_SEASONS
    ),
)

ColumnNotFoundError: unable to find column "goals_scored_per90_rolling_10"; valid columns: ["season", "gw", "element", "opponent", "goals_per_90", "goals_scored", "minutes", "is_home", "is_defender", "is_midfielder", "is_forward", "has_no_form", "goals_scored_per90_rolling_5", "xg_per90_rolling_5", "xa_per90_rolling_5", "expected_pen_attempts_per_90", "xg_for_rolling_5", "goals_for_rolling_5", "xg_against_rolling_5", "goals_against_rolling_5", "team_elo", "opponent_elo", "elo_diff"]

Did you mean "goals_scored_per90_rolling_5"?